In [2]:
from propp_fr import load_models
from rich.jupyter import display

spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


propp_fr package loaded successfully.
Loading models...
CUDA is required, Spacy model should run on GPU.
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH/final_model.pkl
Model Loaded Successfully from local path: C:\Users\covre\PycharmProjects\balzac_comedie_humaine\AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER/final_model

Models Loaded Successfully:
Spacy: fr_dep_news_trf
Mentions Detection: AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH
Coreference Resolution: AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER


In [3]:
from propp_fr import load_text_file
text_content = load_text_file("la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_de_province/balzac-33-FC-rabouilleuse/balzac-33-FC-rabouilleuse.txt")

In [4]:
from propp_fr import generate_tokens_df
tokens_df = generate_tokens_df(text_content, spacy_model)

Batch Spacy Tokenization:   0%|          | 0/8 [00:00<?, ?it/s]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  12%|█▎        | 1/8 [00:05<00:40,  5.82s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
Batch Spacy Tokenization:  25%|██▌       | 2/8 [00:09<00:26,  4.36s/it]C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\thinc\util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release

In [5]:
tokens_df

,paragraph_ID,sentence_ID,token_ID_within_sentence,token_ID_within_document,word,lemma,byte_onset,byte_offset,POS_tag,dependency_relation,syntactic_head_ID,morph
0,0,0,0,0,Livrable,Livrable,0,8,PROPN,ROOT,0,()
1,0,0,1,1,",",",",8,9,PUNCT,punct,0,()
2,0,0,2,2,Colophon,colophon,10,18,NOUN,nmod,0,"(Gender=Masc, Number=Sing)"
3,1,1,0,3,Issu,issu,19,23,VERB,advcl,15,"(Gender=Masc, Number=Sing, Tense=Past, VerbFor..."
4,1,1,1,4,d’,d’,24,26,ADP,case,6,()
...,...,...,...,...,...,...,...,...,...,...,...,...
130658,1722,4929,0,130658,Paris,Paris,610809,610814,PROPN,ROOT,130658,"(Gender=Masc, Number=Sing)"
130659,1722,4929,1,130659,",",",",610814,610815,PUNCT,punct,130658,()
130660,1722,4929,2,130660,novembre,novembre,610816,610824,NOUN,nmod,130658,"(Gender=Masc, Number=Sing)"
130661,1722,4929,3,130661,1842,1842,610825,610829,NUM,nmod,130660,(NumType=Card)


In [6]:
from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

# Load the tokenizer and pre-trained embedding model
tokenizer, embedding_model = load_tokenizer_and_embedding_model(
    mentions_detection_model["base_model_name"],
  )

# Generate embeddings for all tokens
tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
    text_content,
    tokens_df,
    tokenizer,
    embedding_model,
  )

Some weights of CamembertModel were not initialized from the model checkpoint at almanach/camembert-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded encoder model: almanach/camembert-large


Token indices sequence length is longer than the specified maximum sequence length for this model (152035 > 512). Running this sequence through the model will result in indexing errors


In [7]:
from propp_fr import generate_entities_df

entities_df = generate_entities_df(
    tokens_df,
    tokens_embedding_tensor,
    mentions_detection_model,
)

In [8]:
from propp_fr import add_features_to_entities

entities_df = add_features_to_entities(entities_df, tokens_df)

In [9]:
from propp_fr import perform_coreference

entities_df = perform_coreference(
    entities_df,
    tokens_embedding_tensor,
    coreference_resolution_model,
    )

Generating Mention Pairs Features Array...
Postprocessing Mention Pairs Predictions...


C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\propp_fr\propp_fr_coreference_resolution_module.py:1137: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  mention_pairs_df["confidence"].fillna(1, inplace=True)
Generating coreference matrix:  99%|█████████▉| 402403/407402 [00:14<00:00, 17211.92it/s]

402980	Illegal Coreference [16405, 16410]


Generating coreference matrix:  99%|█████████▉| 404554/407402 [00:14<00:00, 9474.72it/s] 

404364	Illegal Coreference [1306, 1573]
404941	Illegal Coreference [2764, 2766]
405593	Illegal Coreference [3442, 3725]


Generating coreference matrix: 100%|█████████▉| 406181/407402 [00:15<00:00, 5054.89it/s]

406734	Illegal Coreference [12193, 12200]


407274	Illegal Coreference [16054, 16130]
407325	Illegal Coreference [9382, 9615]


In [10]:
from propp_fr import extract_attributes
tokens_df = extract_attributes(entities_df, tokens_df)

In [11]:
import pandas as pd
pd.set_option('display.max_columns', None)
tokens_df

,paragraph_ID,sentence_ID,token_ID_within_sentence,token_ID_within_document,word,lemma,byte_onset,byte_offset,POS_tag,dependency_relation,syntactic_head_ID,morph,is_mention_head,char_att_agent,char_att_patient,char_att_mod,char_att_poss
0,0,0,0,0,Livrable,Livrable,0,8,PROPN,ROOT,0,(),0,-1,-1,-1,-1
1,0,0,1,1,",",",",8,9,PUNCT,punct,0,(),0,-1,-1,-1,-1
2,0,0,2,2,Colophon,colophon,10,18,NOUN,nmod,0,"(Gender=Masc, Number=Sing)",0,-1,-1,-1,-1
3,1,1,0,3,Issu,issu,19,23,VERB,advcl,15,"(Gender=Masc, Number=Sing, Tense=Past, VerbFor...",0,-1,-1,-1,-1
4,1,1,1,4,d’,d’,24,26,ADP,case,6,(),0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130658,1722,4929,0,130658,Paris,Paris,610809,610814,PROPN,ROOT,130658,"(Gender=Masc, Number=Sing)",1,-1,-1,-1,-1
130659,1722,4929,1,130659,",",",",610814,610815,PUNCT,punct,130658,(),0,-1,-1,-1,-1
130660,1722,4929,2,130660,novembre,novembre,610816,610824,NOUN,nmod,130658,"(Gender=Masc, Number=Sing)",1,-1,-1,-1,-1
130661,1722,4929,3,130661,1842,1842,610825,610829,NUM,nmod,130660,(NumType=Card),0,-1,-1,-1,-1


In [12]:
from propp_fr import generate_characters_dict

characters_dict = generate_characters_dict(tokens_df, entities_df)

In [13]:
tokens_df

,paragraph_ID,sentence_ID,token_ID_within_sentence,token_ID_within_document,word,lemma,byte_onset,byte_offset,POS_tag,dependency_relation,syntactic_head_ID,morph,is_mention_head,char_att_agent,char_att_patient,char_att_mod,char_att_poss
0,0,0,0,0,Livrable,livrable,0,8,PROPN,ROOT,0,(),0,-1,-1,-1,-1
1,0,0,1,1,",",",",8,9,PUNCT,punct,0,(),0,-1,-1,-1,-1
2,0,0,2,2,Colophon,colophon,10,18,NOUN,nmod,0,"(Gender=Masc, Number=Sing)",0,-1,-1,-1,-1
3,1,1,0,3,Issu,issu,19,23,VERB,advcl,15,"(Gender=Masc, Number=Sing, Tense=Past, VerbFor...",0,-1,-1,-1,-1
4,1,1,1,4,d’,d’,24,26,ADP,case,6,(),0,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130658,1722,4929,0,130658,Paris,paris,610809,610814,PROPN,ROOT,130658,"(Gender=Masc, Number=Sing)",1,-1,-1,-1,-1
130659,1722,4929,1,130659,",",",",610814,610815,PUNCT,punct,130658,(),0,-1,-1,-1,-1
130660,1722,4929,2,130660,novembre,novembre,610816,610824,NOUN,nmod,130658,"(Gender=Masc, Number=Sing)",1,-1,-1,-1,-1
130661,1722,4929,3,130661,1842,1842,610825,610829,NUM,nmod,130660,(NumType=Card),0,-1,-1,-1,-1
